# Connecting to Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# !sudo apt install unzip

In [ ]:
%cd /content/drive/MyDrive/Colab\ Notebooks/Diaz-RAG

/content/drive/.shortcut-targets-by-id/19MF1gFkXkHcfkK3OLFM88MpHSyzcYOwC/Diaz-RAG


In [ ]:
# %cd trec-2022

/content/drive/.shortcut-targets-by-id/19MF1gFkXkHcfkK3OLFM88MpHSyzcYOwC/Diaz-RAG


In [ ]:
# !wget https://data.boisestate.edu/library/Ekstrand/TRECFairRanking/corpus/trec_corpus_20220301_plain.json.gz

In [ ]:
# !git clone https://github.com/kimdanny/Fair-RAG.git

In [ ]:
# !du -sh ./Fair-RAG/

In [ ]:
# !unzip /content/drive/MyDrive/Colab\ Notebooks/Diaz-RAG/Fair-RAG/data/lamp_utility_labels_flanT5Base.zip -d ./Fair-RAG/

# dataset creation

In [ ]:
!pip install ir-datasets

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.0/859.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.0/135.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.5 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=9f630dd686f25aa0edf15de226e6397ddd072b5ec0ee0f4f29065d80f4d1c446
  Stored in directory: /root/.cache/pip/wheels/63/f9/dc/2dd16d3330e327236e4d407941975c42d5159d200cdb7922d8
  Created wheel for cbor: filename=cbor-1.0.0-cp311-cp311-linux_x86_64.whl size=53930 sha256=66ea7b6693c1b5996e192ab7d2719a42cc0138fd3e40028775bcc92057ba6e79
  Stored in directory: /root/.cache/pip/wheels/21/6b/45/0c34253b1af07d1d9dc524f6d44d74a6b191c43152e6aaf641
Successfully built warc3-wet-clueweb09 cbor


In [ ]:
# this cell is to constuct a dataframe which contains two columns query_id and rel_doc_id_list.

import ir_datasets
import pandas as pd

query_rel_dict = dict()

dataset = ir_datasets.load("trec-fair/2022/train")
for i, qrel in enumerate(dataset.qrels_iter()):
    # qrel # namedtuple<query_id, doc_id, relevance, iteration>
    if qrel.relevance==1:
        if qrel.query_id not in query_rel_dict:
            query_rel_dict[qrel.query_id] = []
        query_rel_dict[qrel.query_id].append(qrel.doc_id)

# Convert the dictionary to a list of rows
rows = [{"query_id": k, "rel_doc_id_list": v} for k, v in query_rel_dict.items()]

# Create a DataFrame
qrel_df = pd.DataFrame(rows, columns=["query_id", "rel_doc_id_list"])

# View the DataFrame
print(qrel_df.head())

[INFO] If you have a local copy of https://data.boisestate.edu/library/Ekstrand/TRECFairRanking/2022/trec_2022_train_reldocs.jsonl, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/d132b4cc8c6c75525479728321db5176
[INFO] [starting] https://data.boisestate.edu/library/Ekstrand/TRECFairRanking/2022/trec_2022_train_reldocs.jsonl
[INFO] [finished] https://data.boisestate.edu/library/Ekstrand/TRECFairRanking/2022/trec_2022_train_reldocs.jsonl: [00:01] [18.0MB] [14.4MB/s]


  query_id                                    rel_doc_id_list
0       84  [572, 627, 678, 903, 1193, 1542, 1634, 3751, 3...
1      111  [621, 809, 1380, 6641, 8311, 8937, 13134, 1446...
2      265  [39, 308, 580, 664, 736, 748, 791, 798, 799, 1...
3      323  [849, 852, 1293, 1902, 1942, 2039, 2075, 2082,...
4      396  [344, 676, 808, 872, 1247, 1806, 1828, 2083, 2...


In [ ]:
# this cell is to get id of documents containing less than 250 words.
import gzip
import json
from tqdm import tqdm

desired_doc_ids = []
sum_length = 0
max_body_length_treshold = 300
min_body_length_treshold = 20
min_title_length_treshold = 10
with gzip.open('trec_corpus_20220301_plain.json.gz', 'rt', encoding='utf-8') as f:
    for line in tqdm(f):
        doc = json.loads(line)
        if (len(doc["plain"].split()) <= max_body_length_treshold) and (len(doc["plain"].split()) >= min_body_length_treshold) and (len(doc["title"].split()) >= min_title_length_treshold):
            sum_length += len(doc["plain"].split())
            desired_doc_ids.append(int(doc["id"]))
        # desired_doc_ids.append(doc["id"])

6475537it [12:34, 8580.61it/s]


In [ ]:
len(desired_doc_ids), sum_length/len(desired_doc_ids)

(38423, 103.74871821565209)

In [ ]:
# This cell is to filter out documents not being the list of desired doc ids.
import gzip
import json
from tqdm import tqdm

desired_doc_ids_set = set(desired_doc_ids)  # ensures fast lookup
filtered_desired_doc_ids = []

with gzip.open('./trec_2022_articles_discrete.json.gz', 'rt', encoding='utf-8') as f:
    for line in tqdm(f, desc="Filtering docs"):
        data = json.loads(line)
        if (int(data["page_id"]) in desired_doc_ids_set) and (data["gender_category"] == "Unknown") and (data["first_letter_category"] != "") and (data["creation_date_category"] != "") and (data["years_category"] != "") and (data["relative_pageviews_category"] != ""): #and (data["gender_category"] != "Unknown") and (data["gender_category"] != "")
            filtered_desired_doc_ids.append(data["page_id"])

Filtering docs: 6460238it [01:39, 65023.77it/s]


In [ ]:
621 in desired_doc_ids

False

In [ ]:
len(filtered_desired_doc_ids)

38263

In [ ]:
with open("filtered_desired_doc_ids.txt", "w") as f:
    for doc_id in filtered_desired_doc_ids:
        f.write(str(doc_id) + "\n")

In [ ]:
# Ensure your list of desired IDs is a set for faster lookup
filtered_desired_doc_ids_set = set(filtered_desired_doc_ids)
def count_docs(row):
    counter = 0
    for doc_id in row["rel_doc_id_list"]:
        if int(doc_id) in filtered_desired_doc_ids_set:
            counter += 1
    return counter

# Create a new column to count how many relevant documents per query are in desired_id_set
qrel_df["desired_doc_count"] = qrel_df.apply(count_docs, axis=1)

# Optionally: preview the result
print(qrel_df[["query_id", "desired_doc_count"]].head())

  query_id  desired_doc_count
0       84                  9
1      111                  2
2      265                  5
3      323                 56
4      396                  4


In [ ]:
qrel_df

,query_id,rel_doc_id_list,desired_doc_count
0,84,"[572, 627, 678, 903, 1193, 1542, 1634, 3751, 3...",9
1,111,"[621, 809, 1380, 6641, 8311, 8937, 13134, 1446...",2
2,265,"[39, 308, 580, 664, 736, 748, 791, 798, 799, 1...",5
3,323,"[849, 852, 1293, 1902, 1942, 2039, 2075, 2082,...",56
4,396,"[344, 676, 808, 872, 1247, 1806, 1828, 2083, 2...",4
5,397,"[700, 859, 864, 875, 880, 905, 1006, 1017, 102...",0
6,403,"[308, 340, 700, 728, 736, 784, 851, 852, 892, ...",16
7,409,"[1227, 2995, 3410, 3453, 4193, 4401, 4462, 452...",13
8,426,"[332, 1136, 1756, 2834, 2853, 2956, 3390, 3778...",273
9,475,"[1081, 1581, 1806, 1893, 1938, 2391, 2593, 259...",17


In [ ]:
filtered_desired_doc_ids_set = set(filtered_desired_doc_ids)
final_chosen_documents = set()
for index, row in qrel_df.iterrows():
    if row["desired_doc_count"] >= 5:
        counter = 0
        for doc_id in row["rel_doc_id_list"]:
            if int(doc_id) in filtered_desired_doc_ids_set:
                final_chosen_documents.add(doc_id)
                counter += 1
                if counter >= 100:
                    break

with open("final_chosen_documents.txt", "w") as f:
    for doc_id in final_chosen_documents:
        f.write(str(doc_id) + "\n")

print(len(final_chosen_documents))

1784


In [ ]:
# final_chosen_documents

In [ ]:
import gzip
import json
from tqdm import tqdm
import pandas as pd

corpus_dict = {"docno": [], "title": [], "text": []}
with gzip.open('trec_corpus_20220301_plain.json.gz', 'rt', encoding='utf-8') as f:
    for line in tqdm(f):
        doc = json.loads(line)
        if str(doc["id"]) in final_chosen_documents:
              corpus_dict["docno"].append(str(doc["id"]))
              corpus_dict["text"].append(doc["plain"])
              corpus_dict["title"].append(doc["title"])

corpus_df = pd.DataFrame(corpus_dict)
corpus_df.to_csv("corpus.csv", index=False)

6475537it [06:24, 16828.73it/s]


In [ ]:
import pandas as pd
import gzip
import json
from tqdm import tqdm

selected_ids = set(pd.read_csv("corpus.csv")["docno"].tolist())
selected_ids_set = set(selected_ids)
document_features = pd.DataFrame(columns=["docno", "first_letter_category", "creation_date_category", "years_category", "relative_pageviews_category"])

with gzip.open('./trec_2022_articles_discrete.json.gz', 'rt', encoding='utf-8') as f:
    for line in tqdm(f, desc="Filtering docs"):
        data = json.loads(line)
        if (int(data["page_id"]) in selected_ids):
            document_features.loc[len(document_features)] = [data["page_id"], data["first_letter_category"], data["creation_date_category"], data["years_category"], data["relative_pageviews_category"]]
document_features

Filtering docs: 6460238it [01:47, 60300.15it/s]


,docno,first_letter_category,creation_date_category,years_category,relative_pageviews_category
0,75084,l-r,2001-2006,Unknown,High
1,207255,l-r,2001-2006,Unknown,Medium-High
2,210547,a-d,2001-2006,21st century,Low
3,263287,l-r,2001-2006,21st century,Low
4,268213,l-r,2001-2006,20th century,Medium-Low
...,...,...,...,...,...
1779,68816241,a-d,2017-2022,Unknown,Low
1780,68902183,l-r,2017-2022,Unknown,Low
1781,69435268,l-r,2017-2022,Unknown,Medium-Low
1782,69517985,a-d,2017-2022,Unknown,Low


In [ ]:
document_features.to_csv("document_features.csv", index=False)

# pyterrier indexing

In [ ]:
!apt-get install -y openjdk-11-jdk
!pip install python-terrier

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-dejavu-core fonts-dejavu-extra libatk-wrapper-java
  libatk-wrapper-java-jni libxt-dev libxtst6 libxxf86dga1 openjdk-11-jre
  x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm mesa-utils
The following NEW packages will be installed:
  fonts-dejavu-core fonts-dejavu-extra libatk-wrapper-java
  libatk-wrapper-java-jni libxt-dev libxtst6 libxxf86dga1 openjdk-11-jdk
  openjdk-11-jre x11-utils
0 upgraded, 10 newly installed, 0 to remove and 35 not upgraded.
Need to get 6,920 kB of archives.
After this operation, 16.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-dejavu-core all 2.37-2build1 [1,041 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-dejavu-extra all 2.37-2build1 [2,041 kB]
Get:3 http://archive.ubuntu.com/ubuntu jam

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

In [ ]:
import pyterrier as pt
if not pt.started():
    pt.init()

<ipython-input-6-a9775fa13689>:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...
Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...
Done


Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
<ipython-input-6-a9775fa13689>:3: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [ ]:
import pandas as pd

!rm -rf ./pd_index
df = pd.read_csv("corpus.csv")
df['docno'] = df['docno'].astype(str)
iter_indexer = pt.IterDictIndexer("./pd_index", stemmer=pt.TerrierStemmer.porter, stopwords=pt.TerrierStopwords.terrier, meta={'docno': 40})
indexref = iter_indexer.index(df.to_dict(orient='records'))

In [ ]:
!ls

corpus.csv		      pd_index
Fair-RAG		      qrel_df.csv
filtered_desired_doc_ids.txt  trec_2022_articles_discrete.json.gz
final_chosen_documents.txt    trec_corpus_20220301_plain.json.gz


In [ ]:
import os
print(os.path.exists("./pd_index/data.properties"))
print(os.listdir("./pd_index"))

True
['data.direct.bf', 'data.document.fsarrayfile', 'data.meta.zdata', 'data.meta.idx', 'data.properties', 'data.meta-0.fsomapfile', 'data.inverted.bf', 'data.lexicon.fsomapfile', 'data.lexicon.fsomapid', 'data.lexicon.fsomaphash']


In [ ]:
import pyterrier as pt
import pandas as pd

# Initialize PyTerrier
if not pt.java.started():
    pt.init()

# Load the index
index = pt.IndexFactory.of("./pd_index/data.properties")  # Make sure this path is correct
# print(index.getCollectionStatistics().toString())

# Use the current recommended Retriever class
BM25_r = pt.terrier.Retriever(index, num_results=100, wmodel="BM25")
query_text = "Bernardus Accama was an 18th-century Dutch historical and portrait painter."
query_df = pd.DataFrame([{"qid": "1", "query": query_text}])
res = BM25_r.transform(query_df)
print(res.shape)
# Format the query in a DataFrame

# pt.terrier.Retriever(index).search("mathematical")

# # Display results
# print(results.head())

(32, 6)


In [ ]:
query_df

,qid,query
0,1,Bernardus Accama (1697–1756) was an 18th-centu...


In [ ]:
# import gzip
# import json

# with gzip.open('./trec_2022_articles_discrete.json.gz', 'rt', encoding='utf-8') as f:
#     first_line = f.readline()
#     print(first_line)

{"page_id": 12, "pred_qual": 0.72351444, "qual_cat": "C", "page_countries": [], "page_subcont_regions": [], "source_countries": {"UNK": 52, "United States of America": 41, "United Kingdom": 38, "Canada": 3}, "source_subcont_regions": {"UNK": 52, "Northern America": 44, "Northern Europe": 38}, "gender": [], "occupations": [], "years": [], "num_sitelinks": 142, "relative_pageviews": 1.0, "first_letter": "A", "creation_date": "2001-10-11", "first_letter_category": "a-d", "gender_category": "Unknown", "creation_date_category": "2001-2006", "years_category": "Unknown", "relative_pageviews_category": "High", "num_sitelinks_category": "5+ languages"}



# Retriever

## BM25

In [ ]:
# Adapted from https://github.com/dorianbrown/rank_bm25/blob/master/rank_bm25.py
#!/usr/bin/env python

import math
import numpy as np
from multiprocessing import Pool, cpu_count

"""
All of these algorithms have been taken from the paper:
Trotmam et al, Improvements to BM25 and Language Models Examined

Here we implement all the BM25 variations mentioned.
"""


class BM25:
    def __init__(self, corpus, tokenizer=None):
        self.corpus_size = 0
        self.avgdl = 0
        self.doc_freqs = []
        self.idf = {}
        self.doc_len = []
        self.tokenizer = tokenizer

        if tokenizer:
            corpus = self._tokenize_corpus(corpus)

        nd = self._initialize(corpus)
        self._calc_idf(nd)

    def _initialize(self, corpus):
        nd = {}  # word -> number of documents with word
        num_doc = 0
        for document in corpus:
            self.doc_len.append(len(document))
            num_doc += len(document)

            frequencies = {}
            for word in document:
                if word not in frequencies:
                    frequencies[word] = 0
                frequencies[word] += 1
            self.doc_freqs.append(frequencies)

            for word, freq in frequencies.items():
                try:
                    nd[word] += 1
                except KeyError:
                    nd[word] = 1

            self.corpus_size += 1

        self.avgdl = num_doc / self.corpus_size
        return nd

    def _tokenize_corpus(self, corpus):
        pool = Pool(cpu_count())
        tokenized_corpus = pool.map(self.tokenizer, corpus)
        return tokenized_corpus

    def _calc_idf(self, nd):
        raise NotImplementedError()

    def get_scores(self, query):
        raise NotImplementedError()

    def get_batch_scores(self, query, doc_ids):
        raise NotImplementedError()

    def get_top_n(self, query, documents, n=5):

        assert self.corpus_size == len(
            documents
        ), "The documents given don't match the index corpus!"

        scores = self.get_scores(query)
        top_n = np.argsort(scores)[::-1][:n]
        return [documents[i] for i in top_n]

    # return with scores
    def get_top_n_with_scores(self, query, documents, n=5):

        assert self.corpus_size == len(
            documents
        ), "The documents given don't match the index corpus!"

        scores = self.get_scores(query)
        top_n = np.argsort(scores)[::-1][:n]
        return [(documents[i], scores[i]) for i in top_n]


class BM25Okapi(BM25):
    def __init__(self, corpus, tokenizer=None, k1=1.5, b=0.75, epsilon=0.25):
        self.k1 = k1
        self.b = b
        self.epsilon = epsilon
        super().__init__(corpus, tokenizer)

    def _calc_idf(self, nd):
        """
        Calculates frequencies of terms in documents and in corpus.
        This algorithm sets a floor on the idf values to eps * average_idf
        """
        # collect idf sum to calculate an average idf for epsilon value
        idf_sum = 0
        # collect words with negative idf to set them a special epsilon value.
        # idf can be negative if word is contained in more than half of documents
        negative_idfs = []
        for word, freq in nd.items():
            idf = math.log(self.corpus_size - freq + 0.5) - math.log(freq + 0.5)
            self.idf[word] = idf
            idf_sum += idf
            if idf < 0:
                negative_idfs.append(word)
        self.average_idf = idf_sum / len(self.idf)

        eps = self.epsilon * self.average_idf
        for word in negative_idfs:
            self.idf[word] = eps

    def get_scores(self, query):
        """
        The ATIRE BM25 variant uses an idf function which uses a log(idf) score. To prevent negative idf scores,
        this algorithm also adds a floor to the idf value of epsilon.
        See [Trotman, A., X. Jia, M. Crane, Towards an Efficient and Effective Search Engine] for more info
        :param query:
        :return:
        """
        score = np.zeros(self.corpus_size)
        doc_len = np.array(self.doc_len)
        for q in query:
            q_freq = np.array([(doc.get(q) or 0) for doc in self.doc_freqs])
            score += (self.idf.get(q) or 0) * (
                q_freq
                * (self.k1 + 1)
                / (q_freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl))
            )
        return score

    def get_batch_scores(self, query, doc_ids):
        """
        Calculate bm25 scores between query and subset of all docs
        """
        assert all(di < len(self.doc_freqs) for di in doc_ids)
        score = np.zeros(len(doc_ids))
        doc_len = np.array(self.doc_len)[doc_ids]
        for q in query:
            q_freq = np.array([(self.doc_freqs[di].get(q) or 0) for di in doc_ids])
            score += (self.idf.get(q) or 0) * (
                q_freq
                * (self.k1 + 1)
                / (q_freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl))
            )
        return score.tolist()

In [ ]:
!ls

Fair-RAG  main.ipynb  news  stopword-list.txt  test_questions.json  trec-2022


In [ ]:
!apt-get install -y openjdk-11-jdk
!pip install python-terrier

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

import pyterrier as pt
if not pt.started():
    pt.init()

/tmp/ipython-input-6-1041589085.py:5: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...
Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...
Done


Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipython-input-6-1041589085.py:6: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [ ]:
import re
import pandas as pd

stopwords = set()
with open("./stopword-list.txt", "r", encoding="utf-8") as f:
    for line in f:
        stopwords.add(line.strip())

stemmer = pt.TerrierStemmer.porter
def preprocess(doc):
    cleaned_text = doc.replace("\n", " ").replace("\r", " ")
    clean_text = re.sub(r'\s+', ' ', cleaned_text)
    clean_text = clean_text.split()
    temp_clean_text = []
    for word in clean_text:
        if word not in stopwords:
            temp_clean_text.append(stemmer.stem(word))

    clean_text = " ".join(temp_clean_text)
    return clean_text

task = "News" #"TREC" "X"
if task == "TREC":
    corpus_path = "./trec-2022/corpus.csv"
    corpus = pd.read_csv(corpus_path)
    corpus["document"] = corpus["title"] + "\n" + corpus["text"]
    corpus["processed_text"] = corpus["text"].apply(preprocess)
    corpus["processed_title"] = corpus["title"].apply(preprocess)
    corpus["processed_document"] = corpus["processed_title"] + "\n" + corpus["processed_text"]
    document_list = corpus["document"].tolist()

    documents = corpus[["docno", "document"]].to_dict(orient='records')

    tokenized_corpus = [x.split() for x in document_list]
    bm25 = BM25Okapi(tokenized_corpus)
elif task == "News":
    corpus_path = "./news/news_collection.csv"
    corpus = pd.read_csv(corpus_path)
    corpus["document"] = corpus["title"] + "\n" + corpus["content"]
    corpus["processed_content"] = corpus["content"].apply(preprocess)
    corpus["processed_title"] = corpus["title"].apply(preprocess)
    corpus["processed_document"] = corpus["processed_title"] + "\n" + corpus["processed_content"]
    document_list = corpus["document"].tolist()

    documents = corpus[["docno", "document"]].to_dict(orient='records')

    tokenized_corpus = [x.split() for x in document_list]
    bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
from tqdm import tqdm
import re
import pandas

stopwords = set()
with open("./stopword-list.txt", "r", encoding="utf-8") as f:
    for line in f:
        stopwords.add(line.strip())

stemmer = pt.TerrierStemmer.porter
def preprocess(doc):
    cleaned_text = doc.replace("\n", " ").replace("\r", " ")
    clean_text = re.sub(r'\s+', ' ', cleaned_text)
    clean_text = clean_text.split()
    temp_clean_text = []
    for word in clean_text:
        if word not in stopwords:
            temp_clean_text.append(stemmer.stem(word))

    clean_text = " ".join(temp_clean_text)
    return clean_text

bm25_result = []
task = "News" #"TREC" "X"
if task == "TREC":
    for eachrow in tqdm(corpus.itertuples(index=False)):
        query = eachrow.processed_document
        tokenized_query = query.split()
        selected_profs_with_scores = bm25.get_top_n_with_scores(tokenized_query, documents, n=11)
        related_doc_ids = []
        for i in range(1, 11):
            related_doc_ids.append(selected_profs_with_scores[i][0]["docno"])
        bm25_result.append(related_doc_ids)
elif task == "News":
    query_df = pd.read_csv("./news/news_query_set.csv")
    for eachrow in tqdm(query_df.itertuples(index=False)):
        query = eachrow.title
        query_docno = eachrow.docno
        preprocessed_query = preprocess(query)
        tokenized_query = preprocessed_query.split()
        selected_profs_with_scores = bm25.get_top_n_with_scores(tokenized_query, documents, n=11)
        related_doc_ids = []
        for i in range(11):
            if str(query_docno) == str(selected_profs_with_scores[i][0]["docno"]):
                continue
            related_doc_ids.append(selected_profs_with_scores[i][0]["docno"])
            if len(related_doc_ids) == 10:
                break
        if len(related_doc_ids) < 10:
            print(f"something is wrong with this query: {query_docno}")

        bm25_result.append(related_doc_ids)

612it [02:03,  4.94it/s]


In [ ]:
task = "News" #"TREC" "X"
if task == "TREC":
    corpus['rel_docno'] = bm25_result
    corpus.to_csv("corpus_with_bm25.csv", index=False)
elif task == "News":
    query_df["rel_docno"] = bm25_result
    query_df.to_csv("./news/news_query_set_with_bm25.csv", index=False)

In [ ]:
# corpus

In [ ]:
import json
from tqdm import tqdm
import re
import pandas as pd
import ast

task = "News" #"X"
if task == "News":
    query_set_with_bm25 = pd.read_csv("./news/news_query_set_with_bm25.csv")

def simple_preprocess(text):
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    return text

input_list = []
output_data_dict = {"task":"", "golds": []}
output_data_dict["task"] = "LaMP_5"

for i, eachrow in tqdm(enumerate(query_set_with_bm25.itertuples(index=False))):
    output_inner_dict = {"id":"", "output": ""}
    output_inner_dict["id"] = str(eachrow.docno)
    output_inner_dict["output"] = simple_preprocess(eachrow.content)
    output_data_dict["golds"].append(output_inner_dict)

    input_data_dict = {"id":"", "input":"", "profile": []}
    input_data_dict["id"] = str(eachrow.docno)
    input_data_dict["input"] = "Generate a news for the following news title: " + simple_preprocess(eachrow.title)
    rel_docno_list = ast.literal_eval(eachrow.rel_docno)
    for doc_id in rel_docno_list:
        input_inner_dict = {"title":"", "abstract": "", "id": ""}
        input_inner_dict["title"] = simple_preprocess(corpus[corpus['docno'] == doc_id]["title"].to_list()[0])
        input_inner_dict["abstract"] = simple_preprocess(corpus[corpus['docno'] == doc_id]["content"].to_list()[0])
        input_inner_dict["id"] = str(doc_id)
        input_data_dict["profile"].append(input_inner_dict)
    input_list.append(input_data_dict)

# Save to JSON file
with open("./news/5_user_dev_inputs.json", "w", encoding="utf-8") as f:
    json.dump(input_list, f, indent=2, ensure_ascii=False)

# Save to JSON file
with open("./news/5_user_dev_outputs.json", "w", encoding="utf-8") as f:
    json.dump(output_data_dict, f, indent=2, ensure_ascii=False)

612it [00:55, 11.06it/s]


In [ ]:
# import json
# from tqdm import tqdm
# import re
# import pandas as pd
# import ast

# corpus = pd.read_csv("corpus_with_bm25.csv")

# def simple_preprocess(text):
#     text = text.replace("\r", " ").replace("\n", " ")
#     text = re.sub(r'\s+', ' ', text)
#     return text

# input_list = []
# output_data_dict = {"task":"", "golds": []}
# output_data_dict["task"] = "LaMP_5"

# for i, eachrow in tqdm(enumerate(corpus.itertuples(index=False))):
#     output_inner_dict = {"id":"", "output": ""}
#     output_inner_dict["id"] = str(eachrow.docno)
#     output_inner_dict["output"] = simple_preprocess(eachrow.title)
#     output_data_dict["golds"].append(output_inner_dict)

#     input_data_dict = {"id":"", "input":"", "profile": []}
#     input_data_dict["id"] = str(eachrow.docno)
#     input_data_dict["input"] = "Generate a title for the following wikipedia document: " + simple_preprocess(eachrow.text)
#     rel_docno_list = ast.literal_eval(eachrow.rel_docno)
#     for doc_id in rel_docno_list:
#         input_inner_dict = {"title":"", "abstract": "", "id": ""}
#         input_inner_dict["title"] = simple_preprocess(corpus[corpus['docno'] == doc_id]["title"].to_list()[0])
#         input_inner_dict["abstract"] = simple_preprocess(corpus[corpus['docno'] == doc_id]["text"].to_list()[0])
#         input_inner_dict["id"] = str(doc_id)
#         input_data_dict["profile"].append(input_inner_dict)
#     input_list.append(input_data_dict)
#     if i == 1499:
#         break

# # Save to JSON file
# with open("5_user_dev_inputs.json", "w", encoding="utf-8") as f:
#     json.dump(input_list, f, indent=2, ensure_ascii=False)

# # Save to JSON file
# with open("5_user_dev_outputs.json", "w", encoding="utf-8") as f:
#     json.dump(output_data_dict, f, indent=2, ensure_ascii=False)

1499it [00:11, 128.45it/s]


## Contriver

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForMaskedLM

In [ ]:
from tqdm import tqdm
def mean_pooling(token_embeddings, mask):
    token_embeddings = token_embeddings.masked_fill(~mask[..., None].bool(), 0.0)
    sentence_embeddings = token_embeddings.sum(dim=1) / mask.sum(dim=1)[..., None]
    return sentence_embeddings

def batchify(lst, batch_size):
    return [lst[i : i + batch_size] for i in range(0, len(lst), batch_size)]

@torch.no_grad()
def encode_corpus(contriver, tokenizer, corpus, batch_size=16):
    counter = 0
    for batch in tqdm(batchify(corpus, batch_size)):
        tokens = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to("cuda:0")
        outputs = contriver(**tokens)
        embeddings = mean_pooling(outputs.last_hidden_state, tokens["attention_mask"])
        torch.save(embeddings, f"./news/embeddings/batch_{counter}.pt")
        counter += 1


@torch.no_grad()
def retrieve_top_k_with_contriver(contriver, tokenizer, profile, query, k, number_of_batches) -> list[tuple]:
    """
    Uses precomputed corpus_embeddings to retrieve top-k documents for a given query.
    """
    # Compute query embedding
    query_tokens = tokenizer([query], padding=True, truncation=True, return_tensors="pt").to("cuda:0")
    output_query = contriver(**query_tokens)
    query_embedding = mean_pooling(output_query.last_hidden_state, query_tokens["attention_mask"])  # (1, dim)
    scores = []
    for i in range(number_of_batches):
        batch_embeddings = torch.load(f"./news/embeddings/batch_{i}.pt").to("cuda:0")
        temp_scores = query_embedding.squeeze() @ batch_embeddings.T
        scores.extend(temp_scores.tolist())

    topk_values, topk_indices = torch.topk(torch.tensor(scores), k)
    topk_values = topk_values.tolist()
    topk_indices = topk_indices.tolist()
    return [(profile[i], score) for score, i in zip(topk_values, topk_indices)]

In [ ]:
from tqdm import tqdm
import re
import pandas as pd

def preprocess(doc):
    cleaned_text = doc.replace("\n", " ").replace("\r", " ")
    clean_text = re.sub(r'\s+', ' ', cleaned_text)
    return clean_text

bm25_result = []
task = "News" #"TREC" "X"
if task == "TREC":
    pass
elif task == "News":
    corpus_path = "./news/news_collection.csv"
    corpus = pd.read_csv(corpus_path)
    corpus["document"] = corpus["title"] + "\n" + corpus["content"]
    corpus["processed_content"] = corpus["content"].apply(preprocess)
    corpus["processed_title"] = corpus["title"].apply(preprocess)
    corpus["processed_document"] = corpus["processed_title"] + "\n" + corpus["processed_content"]
    document_list = corpus["document"].tolist()
    profile = corpus[["docno", "processed_document"]].to_dict(orient='records')

In [ ]:
# del corpus

In [ ]:
!ls

In [ ]:
from tqdm import tqdm
import re
import pandas

contriver_checkpoint = "facebook/contriever"
tokenizer = AutoTokenizer.from_pretrained(contriver_checkpoint)
contriver = AutoModel.from_pretrained(contriver_checkpoint).to("cuda:0")
contriver.eval()

In [ ]:
batch_size = 256
encode_corpus(contriver, tokenizer, document_list, batch_size)

In [ ]:
def preprocess(doc):
    cleaned_text = doc.replace("\n", " ").replace("\r", " ")
    clean_text = re.sub(r'\s+', ' ', cleaned_text)
    return clean_text

contriver_result = []
task = "News" #"TREC" "X"
if task == "TREC":
    pass
elif task == "News":
    query_df = pd.read_csv("./news/news_query_set.csv")
    for eachrow in tqdm(query_df.itertuples(index=False)):
        query = eachrow.title
        query_docno = eachrow.docno
        preprocessed_query = preprocess(query)
        selected_profs_with_scores = retrieve_top_k_with_contriver(contriver, tokenizer, profile, preprocessed_query, 11, 147)
        related_doc_ids = []
        for i in range(11):
            if str(query_docno) == str(selected_profs_with_scores[i][0]["docno"]):
                continue
            related_doc_ids.append(selected_profs_with_scores[i][0]["docno"])
            if len(related_doc_ids) == 10:
                break
        if len(related_doc_ids) < 10:
            print(f"something is wrong with this query: {query_docno}")

        contriver_result.append(related_doc_ids)

612it [06:47,  1.50it/s]


In [ ]:
task = "News" #"TREC" "X"
if task == "TREC":
    pass
elif task == "News":
    query_df["rel_docno"] = contriver_result
    query_df.to_csv("./news/news_query_set_with_contriver.csv", index=False)

In [ ]:
import json
from tqdm import tqdm
import re
import pandas as pd
import ast

task = "News" #"X"
if task == "News":
    query_set_with_contriver = pd.read_csv("./news/news_query_set_with_contriver.csv")

def simple_preprocess(text):
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    return text

input_list = []
output_data_dict = {"task":"", "golds": []}
output_data_dict["task"] = "LaMP_5"

for i, eachrow in tqdm(enumerate(query_set_with_contriver.itertuples(index=False))):
    output_inner_dict = {"id":"", "output": ""}
    output_inner_dict["id"] = str(eachrow.docno)
    output_inner_dict["output"] = simple_preprocess(eachrow.content)
    output_data_dict["golds"].append(output_inner_dict)

    input_data_dict = {"id":"", "input":"", "profile": []}
    input_data_dict["id"] = str(eachrow.docno)
    input_data_dict["input"] = "Generate a news for the following news title: " + simple_preprocess(eachrow.title)
    rel_docno_list = ast.literal_eval(eachrow.rel_docno)
    for doc_id in rel_docno_list:
        input_inner_dict = {"title":"", "abstract": "", "id": ""}
        input_inner_dict["title"] = simple_preprocess(corpus[corpus['docno'] == doc_id]["title"].to_list()[0])
        input_inner_dict["abstract"] = simple_preprocess(corpus[corpus['docno'] == doc_id]["content"].to_list()[0])
        input_inner_dict["id"] = str(doc_id)
        input_data_dict["profile"].append(input_inner_dict)
    input_list.append(input_data_dict)

# Save to JSON file
with open("./news/contriver/5_user_dev_inputs.json", "w", encoding="utf-8") as f:
    json.dump(input_list, f, indent=2, ensure_ascii=False)

# Save to JSON file
with open("./news/contriver/5_user_dev_outputs.json", "w", encoding="utf-8") as f:
    json.dump(output_data_dict, f, indent=2, ensure_ascii=False)

612it [01:23,  7.30it/s]


## SPLADE

In [ ]:
!pip install faiss-cpu==1.7.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 67.1 MB/s eta 0:00:00


In [ ]:
!pip install sparsembed==0.1.1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 52.0 MB/s eta 0:00:00
  Created wheel for sparsembed: filename=sparsembed-0.1.1-py3-none-any.whl size=25484 sha256=5119377f6c5fa5f3731c9fc2a15e023cb0f740cde7b2b279ac877f62ed2bbffa
  Sto

In [ ]:
from sparsembed import model, retrieve
from transformers import AutoModel, AutoTokenizer, AutoModelForMaskedLM

In [ ]:
from tqdm import tqdm
import re
import pandas as pd

def preprocess(doc):
    cleaned_text = doc.replace("\n", " ").replace("\r", " ")
    clean_text = re.sub(r'\s+', ' ', cleaned_text)
    return clean_text

bm25_result = []
task = "News" #"TREC" "X"
if task == "TREC":
    pass
elif task == "News":
    corpus_path = "./news/news_collection.csv"
    corpus = pd.read_csv(corpus_path)
    corpus["processed_content"] = corpus["content"].apply(preprocess)
    corpus["processed_title"] = corpus["title"].apply(preprocess)
    corpus["text"] = corpus["processed_title"] + "\n" + corpus["processed_content"]
    splade_corpus = [
            {"id": eachrow.docno, "text": eachrow.text} for eachrow in corpus.itertuples(index=False)
        ]

In [ ]:
def retrieve_top_k_with_splade(retriever, query, k, batch_size=16):
    output = retriever(
        [query],  # can pass multiple queries but passing a single query here
        # k_tokens=20,
        k=k,  # Number of documents to retrieve.
        batch_size=batch_size,
    )
    return [({"id": x["id"]}, x["similarity"]) for x in output[0]]

In [ ]:
splade_checkpoint = "naver/splade_v2_max"
splade_model = model.Splade(model=AutoModelForMaskedLM.from_pretrained(splade_checkpoint).to("cuda:0"),
                           tokenizer=AutoTokenizer.from_pretrained(splade_checkpoint),device="cuda:0",)
batch_size = 8
retriever = retrieve.SpladeRetriever(
    key="id",  # Key identifier of each document
    on=["text"],  # Fields to search.
    model=splade_model,
    )
retriever = retriever.add(documents=splade_corpus, batch_size=batch_size) # k_tokens=256 # Number of activated tokens.

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/488 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

100%|██████████| 4695/4695 [08:42<00:00,  8.98it/s]


In [ ]:
def preprocess(doc):
    cleaned_text = doc.replace("\n", " ").replace("\r", " ")
    clean_text = re.sub(r'\s+', ' ', cleaned_text)
    return clean_text

splade_result = []
task = "News" #"TREC" "X"
if task == "TREC":
    pass
elif task == "News":
    batch_size = 8
    query_df = pd.read_csv("./news/news_query_set.csv")
    for eachrow in tqdm(query_df.itertuples(index=False)):
        query = eachrow.title
        query_docno = eachrow.docno
        preprocessed_query = preprocess(query)
        selected_profs_with_scores = retrieve_top_k_with_splade(retriever, query, 11, batch_size)
        related_doc_ids = []
        for i in range(11):
            if str(query_docno) == str(selected_profs_with_scores[i][0]["id"]):
                continue
            related_doc_ids.append(selected_profs_with_scores[i][0]["id"])
            if len(related_doc_ids) == 10:
                break
        if len(related_doc_ids) < 10:
            print(f"something is wrong with this query: {query_docno}")

        splade_result.append(related_doc_ids)

100%|██████████| 1/1 [00:00<00:00,  8.21it/s]
612it [00:51, 11.90it/s]


In [ ]:
task = "News" #"TREC" "X"
if task == "TREC":
    pass
elif task == "News":
    query_df["rel_docno"] = splade_result
    query_df.to_csv("./news/news_query_set_with_splade.csv", index=False)

In [ ]:
import json
from tqdm import tqdm
import re
import pandas as pd
import ast

task = "News" #"X"
if task == "News":
    query_set_with_spalde = pd.read_csv("./news/news_query_set_with_splade.csv")

def simple_preprocess(text):
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    return text

input_list = []
output_data_dict = {"task":"", "golds": []}
output_data_dict["task"] = "LaMP_5"

for i, eachrow in tqdm(enumerate(query_set_with_spalde.itertuples(index=False))):
    output_inner_dict = {"id":"", "output": ""}
    output_inner_dict["id"] = str(eachrow.docno)
    output_inner_dict["output"] = simple_preprocess(eachrow.content)
    output_data_dict["golds"].append(output_inner_dict)

    input_data_dict = {"id":"", "input":"", "profile": []}
    input_data_dict["id"] = str(eachrow.docno)
    input_data_dict["input"] = "Generate a news for the following news title: " + simple_preprocess(eachrow.title)
    rel_docno_list = ast.literal_eval(eachrow.rel_docno)
    for doc_id in rel_docno_list:
        input_inner_dict = {"title":"", "abstract": "", "id": ""}
        input_inner_dict["title"] = simple_preprocess(corpus[corpus['docno'] == doc_id]["title"].to_list()[0])
        input_inner_dict["abstract"] = simple_preprocess(corpus[corpus['docno'] == doc_id]["content"].to_list()[0])
        input_inner_dict["id"] = str(doc_id)
        input_data_dict["profile"].append(input_inner_dict)
    input_list.append(input_data_dict)

# Save to JSON file
with open("./news/splade/5_user_dev_inputs.json", "w", encoding="utf-8") as f:
    json.dump(input_list, f, indent=2, ensure_ascii=False)

# Save to JSON file
with open("./news/splade/5_user_dev_outputs.json", "w", encoding="utf-8") as f:
    json.dump(output_data_dict, f, indent=2, ensure_ascii=False)

612it [00:49, 12.42it/s]


# utility calculation

In [12]:
import warnings
warnings.filterwarnings("ignore")

In [13]:
import warnings
warnings.filterwarnings("ignore", message="You seem to be using the pipelines sequentially on GPU")

In [1]:
!ls

Fair-RAG  main.ipynb  main1.ipynb  main2.ipynb	news  stopword-list.txt


In [5]:
%cd Fair-RAG/

[Errno 2] No such file or directory: 'Fair-RAG/'
/mnt/primary/GroupFairnessConsumption/Fair-RAG


/opt/miniconda3/envs/fairrag/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [6]:
!ls

README.md    expected_exposure	     generator	      requirements.txt
__pycache__  experiment.py	     normalize_eu.py  retrieval
data	     flanT5Base_RAG_output   packages.txt     utility_labels
eval	     flanT5Small_RAG_output  perturbation     utils.py


In [ ]:
# !pip install -q -r requirements.txt

In [10]:
!python -W ignore ./utility_labels/inference.py --model_name gemma2-9b --lamp_num 5 --experiment_baseline --dataset news --retriever bm25 
#> 5_output_baseline.log1 2>&1
#flanT5Small flanT5Base flanT5XXL

612it [04:24,  2.31it/s]


In [ ]:
!python -W ignore ./utility_labels/inference.py --model_name flanT5Base --lamp_num 5 --k 1 --dataset news --retriever splade 
#> 5_output_augment.log1 2>&1
#flanT5Small flanT5Base flanT5XXL

0it [00:00, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (3049 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors
455it [1:49:46, 17.64s/it]

In [18]:
!ls

README.md    expected_exposure	     generator	      requirements.txt
__pycache__  experiment.py	     normalize_eu.py  retrieval
data	     flanT5Base_RAG_output   packages.txt     utility_labels
eval	     flanT5Small_RAG_output  perturbation     utils.py


In [7]:
!python utility_labels/lamp_eval.py --model_name flanT5XL --lamp_num 5 --dataset news --retriever contriver
#flanT5Small flanT5Base flanT5XL

loading data ...
calculating baseline answer qualities ...
calculating augmented answer qualities ...
calculating differences ...


In [ ]:
# !python utility_labels/analyze_delta.py --model_name flanT5Base --lamp_num 5
#flanT5Small

In [ ]:
# !python utility_labels/make_utility_dataset.py --model_name flanT5Base --lamp_num 5
#flanT5Small

1500it [00:00, 80047.02it/s]


# RAG Paradigm

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
%cd Fair-RAG/

/mnt/primary/GroupFairnessConsumption/Fair-RAG


In [ ]:
# !pip install -q -r requirements.txt

In [9]:
!python -W ignore ./utility_labels/inference.py --model_name flanT5Base --lamp_num 5 --k 10 --dataset news --retriever splade 
#> 5_output_rag1.log 2>&1
#flanT5Small flanT5Base

0it [00:00, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (992 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (530 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors
612it [26:19,  2.58s/it]


# Entailment

In [ ]:
!pip install -q transformers

In [ ]:
%cd Fair-RAG/

/content/drive/.shortcut-targets-by-id/19MF1gFkXkHcfkK3OLFM88MpHSyzcYOwC/Diaz-RAG/Fair-RAG


In [ ]:
import pandas as pd

def load_df(fp: str) -> pd.DataFrame:
    # Must read qid and pid as string not as int
    dtype_spec = {"qid": str, "pid": str, "answer": str, "target": str}
    df = pd.read_csv(fp, delimiter="\t", dtype=dtype_spec)
    return df

rag_result = load_df("5_output_rag.log")

In [ ]:
rag_result

,qid,pid,answer,target
0,48479768,-1,Australian cricket team against Pakistan in th...,Australian cricket team against Pakistan in Sr...
1,44425132,-1,2014–15 ISU Speed Skating World Cup – World Cu...,2014–15 ISU Speed Skating World Cup – World Cu...
2,44778334,-1,2014–15 ISU Speed Skating World Cup – World Cu...,2014–15 ISU Speed Skating World Cup – World Cu...
3,10559453,-1,Cycling at the 1984 Summer Olympics – Women's ...,Cycling at the 1984 Summer Olympics – Women's ...
4,49241008,-1,Scottish cricket team in the United Arab Emira...,Scottish cricket team against the Netherlands ...
...,...,...,...,...
585,54667952,-1,2017 Southeast Asian Games – Women's rugby sevens,Rugby sevens at the 2017 Southeast Asian Games...
586,8201829,-1,List of FIS Nordic World Ski Championships med...,List of FIS Nordic World Ski Championships med...
587,474130,-1,List of United States Army Air Force - Air For...,List of United States Army Air Forces Air Forc...
588,46944013,-1,Men's Youth 420 (U19),Sailing at the 2015 Southeast Asian Games – Me...


In [ ]:
from transformers import pipeline, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-mnli")
classifier = pipeline("zero-shot-classification", model="FacebookAI/roberta-large-mnli")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
import json
from tqdm import tqdm

with open('./data/lamp/5_user_dev_inputs.json', 'r') as f:
    data = json.load(f)
entailment_dict = {"doc_id":[], "profile_doc_id":[], "score":[]}
counter = 0
for item in tqdm(data):
    premise = rag_result[rag_result['qid'] == item["id"]]["answer"].tolist()[0]
    # print(premise)
    for context in item["profile"]:
        context_id = int(context["id"])
        context_title = context["title"]
        context_abstract = context["abstract"]
        hypotheses = context_title + "\n" + context_abstract
        if len(tokenizer.tokenize(hypotheses)) + len(tokenizer.tokenize(premise)) > 512:
            # print("*************")
            # print(hypotheses)
            # print("*************")
            hypotheses = tokenizer.tokenize(hypotheses)[:505-len(tokenizer.tokenize(premise))]
            hypotheses = tokenizer.convert_tokens_to_ids(hypotheses)
            hypotheses = tokenizer.decode(hypotheses, skip_special_tokens=True)
            # print(hypotheses)
            # print("*************")
        result = classifier(premise, hypotheses, truncation=True, max_length=512, multi_label=False)
        entailment_dict["doc_id"].append(item["id"])
        entailment_dict["profile_doc_id"].append(context_id)
        entailment_dict["score"].append(result["scores"][0])
    counter += 1
pd.DataFrame(entailment_dict).to_csv("entailment.csv", index=False)

100%|██████████| 590/590 [12:51<00:00,  1.31s/it]


# Group Consumption Evaluation

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
%cd ..

/content/drive/.shortcut-targets-by-id/19MF1gFkXkHcfkK3OLFM88MpHSyzcYOwC/Diaz-RAG


In [ ]:
%cd ./trec-2022

/content/drive/.shortcut-targets-by-id/19MF1gFkXkHcfkK3OLFM88MpHSyzcYOwC/Diaz-RAG/trec-2022


In [ ]:
import pandas as pd

document_features = pd.read_csv("document_features.csv")

In [ ]:
cols = document_features.columns
fl = set(document_features[cols[1]].tolist())
cd = set(document_features[cols[2]].tolist())
y = set(document_features[cols[3]].tolist())
rp = set(document_features[cols[4]].tolist())

In [ ]:
%cd ..
%cd Fair-RAG

/content/drive/.shortcut-targets-by-id/19MF1gFkXkHcfkK3OLFM88MpHSyzcYOwC/Diaz-RAG
/content/drive/.shortcut-targets-by-id/19MF1gFkXkHcfkK3OLFM88MpHSyzcYOwC/Diaz-RAG/Fair-RAG


In [ ]:
import pandas as pd
llm_name = "flanT5Small" # flanT5Small flanT5Base
utility_labels = pd.read_csv(f"./data/lamp_utility_labels_{llm_name}/5_relevance_mapping.tsv", sep='\t')

In [ ]:
import math
import pandas as pd

fl_category_utility = pd.DataFrame(columns=list(fl))
fl_category_entailment = pd.DataFrame(columns=list(fl))
fl_category_exposure = pd.DataFrame(columns=list(fl))

cd_category_utility = pd.DataFrame(columns=list(cd))
cd_category_entailment = pd.DataFrame(columns=list(cd))
cd_category_exposure = pd.DataFrame(columns=list(cd))

y_category_utility = pd.DataFrame(columns=list(y))
y_category_entailment = pd.DataFrame(columns=list(y))
y_category_exposure = pd.DataFrame(columns=list(y))

rp_category_utility = pd.DataFrame(columns=list(rp))
rp_category_entailment = pd.DataFrame(columns=list(rp))
rp_category_exposure = pd.DataFrame(columns=list(rp))

entailment = pd.read_csv(f"./{llm_name}_RAG_output/entailment.csv")
grouped = entailment.groupby('doc_id')

# Iterate through the groups
for group_name, group_df in grouped:
    doc_id = group_name

    fl_category_dict = dict()
    for value in fl:
        fl_category_dict[value] = {"utility":0, "entailment": 0, "exposure": 0, "count": 0}

    cd_category_dict = dict()
    for value in cd:
        cd_category_dict[value] = {"utility":0, "entailment": 0, "exposure": 0, "count": 0}

    y_category_dict = dict()
    for value in y:
        y_category_dict[value] = {"utility":0, "entailment": 0, "exposure": 0, "count": 0}

    rp_category_dict = dict()
    for value in rp:
        rp_category_dict[value] = {"utility":0, "entailment": 0, "exposure": 0, "count": 0}

    rank_position = 1
    for eachrow in group_df.itertuples(index=False):
        profile_doc_id = eachrow.profile_doc_id
        entailment_score = int(eachrow.score > 0.5) #eachrow.score
        utility_score = utility_labels[(utility_labels["qid"] == doc_id) & (utility_labels["pid"] == profile_doc_id)]["relevance_label"].values[0]
        fl_value = document_features[document_features["docno"]==profile_doc_id]["first_letter_category"].values[0]
        fl_category_dict[fl_value]["entailment"] += entailment_score
        fl_category_dict[fl_value]["utility"] += int(utility_score) # (int(utility_score) * (1/(math.log2(rank_position)+1)))
        fl_category_dict[fl_value]["count"] += 1

        cd_value = document_features[document_features["docno"]==profile_doc_id]["creation_date_category"].values[0]
        cd_category_dict[cd_value]["entailment"] += entailment_score
        cd_category_dict[cd_value]["utility"] += int(utility_score) # (int(utility_score) * (1/(math.log2(rank_position)+1)))
        cd_category_dict[cd_value]["count"] += 1

        y_value = document_features[document_features["docno"]==profile_doc_id]["years_category"].values[0]
        y_category_dict[y_value]["entailment"] += entailment_score
        y_category_dict[y_value]["utility"] += int(utility_score) # (int(utility_score) * (1/(math.log2(rank_position)+1)))
        y_category_dict[y_value]["count"] += 1

        rp_value = document_features[document_features["docno"]==profile_doc_id]["relative_pageviews_category"].values[0]
        rp_category_dict[rp_value]["entailment"] += entailment_score
        rp_category_dict[rp_value]["utility"] += int(utility_score) # (int(utility_score) * (1/(math.log2(rank_position)+1)))
        rp_category_dict[rp_value]["count"] += 1

        rank_position += 1

    for key, vals in fl_category_dict.items():
        count = vals['count']
        if count == 0:
            continue
        vals['utility'] /= count
        vals['entailment'] /= count
        vals['exposure'] = count/len(group_df)

    utility_scores = {k: v['utility'] for k, v in fl_category_dict.items()}
    fl_category_utility.loc[len(fl_category_utility)] = utility_scores

    entailment_scores = {k: v['entailment'] for k, v in fl_category_dict.items()}
    fl_category_entailment.loc[len(fl_category_entailment)] = entailment_scores

    exposure_scores = {k: v['exposure'] for k, v in fl_category_dict.items()}
    fl_category_exposure.loc[len(fl_category_exposure)] = exposure_scores

    for key, vals in cd_category_dict.items():
        count = vals['count']
        if count == 0:
            continue
        vals['utility'] /= count
        vals['entailment'] /= count
        vals['exposure'] = count/len(group_df)

    utility_scores = {k: v['utility'] for k, v in cd_category_dict.items()}
    cd_category_utility.loc[len(cd_category_utility)] = utility_scores

    entailment_scores = {k: v['entailment'] for k, v in cd_category_dict.items()}
    cd_category_entailment.loc[len(cd_category_entailment)] = entailment_scores

    exposure_scores = {k: v['exposure'] for k, v in cd_category_dict.items()}
    cd_category_exposure.loc[len(cd_category_exposure)] = exposure_scores

    for key, vals in y_category_dict.items():
        count = vals['count']
        if count == 0:
            continue
        vals['utility'] /= count
        vals['entailment'] /= count
        vals['exposure'] = count/len(group_df)

    utility_scores = {k: v['utility'] for k, v in y_category_dict.items()}
    y_category_utility.loc[len(y_category_utility)] = utility_scores

    entailment_scores = {k: v['entailment'] for k, v in y_category_dict.items()}
    y_category_entailment.loc[len(y_category_entailment)] = entailment_scores

    exposure_scores = {k: v['exposure'] for k, v in y_category_dict.items()}
    y_category_exposure.loc[len(y_category_exposure)] = exposure_scores

    for key, vals in rp_category_dict.items():
        count = vals['count']
        if count == 0:
            continue
        vals['utility'] /= count
        vals['entailment'] /= count
        vals['exposure'] = count/len(group_df)

    utility_scores = {k: v['utility'] for k, v in rp_category_dict.items()}
    rp_category_utility.loc[len(rp_category_utility)] = utility_scores

    entailment_scores = {k: v['entailment'] for k, v in rp_category_dict.items()}
    rp_category_entailment.loc[len(rp_category_entailment)] = entailment_scores

    exposure_scores = {k: v['exposure'] for k, v in rp_category_dict.items()}
    rp_category_exposure.loc[len(rp_category_exposure)] = exposure_scores

In [ ]:
def general_result_analysis(category_utility, category_entailment, category_exposure):
    avg_utility = category_utility.mean(numeric_only=True).to_frame(name='AVG Utility')
    avg_consumption = category_entailment.mean(numeric_only=True).to_frame(name='AVG Consumption')
    avg_exposure = category_exposure.mean(numeric_only=True).to_frame(name='AVG Exposure')
    consumption_utility_ratio = (category_entailment.mean(numeric_only=True)/category_utility.mean(numeric_only=True)).to_frame(name='consumption_to_utility_ratio')
    consumption_exposure_ratio = (category_entailment.mean(numeric_only=True)/category_exposure.mean(numeric_only=True)).to_frame(name='consumption_to_exposure_ratio')
    return pd.concat([avg_utility, avg_consumption, avg_exposure, consumption_utility_ratio, consumption_exposure_ratio], axis=1)

In [ ]:
general_result_analysis(fl_category_utility, fl_category_entailment, fl_category_exposure)

,AVG Utility,AVG Consumption,AVG Exposure,consumption_to_utility_ratio,consumption_to_exposure_ratio
s-,0.215757,0.192805,0.170912,0.893623,1.128098
a-d,0.343109,0.323128,0.404560,0.941767,0.798716
e-k,0.182386,0.200740,0.100314,1.100632,2.001107
l-r,0.325016,0.294135,0.324214,0.904986,0.907225


In [ ]:
general_result_analysis(y_category_utility, y_category_entailment, y_category_exposure)

,AVG Utility,AVG Consumption,AVG Exposure,consumption_to_utility_ratio,consumption_to_exposure_ratio
Pre-1900s,0.024004,0.015252,0.012421,0.635371,1.227848
Unknown,0.327138,0.325758,0.442296,0.995781,0.736517
20th century,0.167504,0.126280,0.114308,0.753894,1.104736
21st century,0.367975,0.334221,0.430975,0.908270,0.775499


In [ ]:
general_result_analysis(cd_category_utility, cd_category_entailment, cd_category_exposure)

,AVG Utility,AVG Consumption,AVG Exposure,consumption_to_utility_ratio,consumption_to_exposure_ratio
2012-2016,0.286973,0.290808,0.297484,1.013365,0.977558
2007-2011,0.454193,0.358115,0.491981,0.788464,0.727904
2017-2022,0.134359,0.163779,0.084591,1.218965,1.936124
2001-2006,0.216438,0.191246,0.125943,0.883603,1.518504


In [ ]:
general_result_analysis(rp_category_utility, rp_category_entailment, rp_category_exposure)

,AVG Utility,AVG Consumption,AVG Exposure,consumption_to_utility_ratio,consumption_to_exposure_ratio
Medium-Low,0.224233,0.142872,0.091824,0.637160,1.555936
Medium-High,0.159822,0.136749,0.072956,0.855631,1.874401
High,0.045860,0.040487,0.014151,0.882857,2.861111
Low,0.497687,0.467720,0.821069,0.939787,0.569647


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def group_wise_average_comparison_plot(category_utility, category_entailment, category_exposure, output_file_name):
    # Calculate average per group for each metric (columns are groups)
    avg_utility = category_utility.mean(numeric_only=True)
    avg_consumption = category_entailment.mean(numeric_only=True)
    avg_exposure = category_exposure.mean(numeric_only=True)

    # Combine into a single DataFrame for easier plotting
    avg_df = pd.DataFrame({
        'Utility': avg_utility,
        'Consumption': avg_consumption,
        'Exposure': avg_exposure
    })

    # Reset index to get groups as a column
    avg_df = avg_df.reset_index().rename(columns={'index': 'Group'})

    # Melt to long format for seaborn grouped barplot
    avg_long = avg_df.melt(id_vars='Group', var_name='Metric', value_name='Average')

    # Set plot style
    sns.set(style='whitegrid')

    # Plot grouped bar chart
    plt.figure(figsize=(10,6))
    sns.barplot(data=avg_long, x='Group', y='Average', hue='Metric', palette='tab10')

    plt.title('Group-wise Average Comparison of Utility, Consumption, and Exposure')
    plt.ylabel('Average Value')
    plt.xlabel('Group')
    plt.legend(title='Metric')
    plt.tight_layout()
    output_path = f'{output_file_name}.png'
    plt.savefig(output_path, dpi=300)  # High-resolution PNG
    plt.close()  # Close to avoid displaying inline if not needed

In [ ]:
group_wise_average_comparison_plot(fl_category_utility, fl_category_entailment, fl_category_exposure, "fl_bar_plot")

In [ ]:
group_wise_average_comparison_plot(y_category_utility, y_category_entailment, y_category_exposure, "y_bar_plot")

In [ ]:
group_wise_average_comparison_plot(cd_category_utility, cd_category_entailment, cd_category_exposure, "cd_bar_plot")

In [ ]:
group_wise_average_comparison_plot(rp_category_utility, rp_category_entailment, rp_category_exposure, "rp_bar_plot")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Helper to reshape to long format
def melt_df(df, value_name):
    df = df.copy()
    df['Ranking'] = df.index
    return df.melt(id_vars='Ranking', var_name='Group', value_name=value_name)

# Main function: 3 side-by-side subplots
def distribution_across_ranking_plot(category_utility, category_entailment, category_exposure, output_file_name):
    # Melt the data
    utility_long = melt_df(category_utility, 'Utility')
    consumption_long = melt_df(category_entailment, 'Consumption')
    exposure_long = melt_df(category_exposure, 'Exposure')

    # Set style
    sns.set(style='whitegrid')

    # Create side-by-side subplots
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 6), sharey=False)

    # Plot Utility
    sns.boxplot(data=utility_long, x='Group', y='Utility', palette='Blues', ax=axes[0])
    axes[0].set_title('Utility Distribution')
    axes[0].set_xlabel('Group')
    axes[0].set_ylabel('Utility')

    # Plot Consumption
    sns.boxplot(data=consumption_long, x='Group', y='Consumption', palette='Greens', ax=axes[1])
    axes[1].set_title('Consumption Distribution')
    axes[1].set_xlabel('Group')
    axes[1].set_ylabel('Consumption')

    # Plot Exposure
    sns.boxplot(data=exposure_long, x='Group', y='Exposure', palette='Oranges', ax=axes[2])
    axes[2].set_title('Exposure Distribution')
    axes[2].set_xlabel('Group')
    axes[2].set_ylabel('Exposure')

    # Adjust layout
    plt.tight_layout()
    output_path = f'{output_file_name}.png'
    plt.savefig(output_path, dpi=300)  # High-resolution PNG
    plt.close()  # Close to avoid displaying inline if not needed
    # plt.show()


In [ ]:
distribution_across_ranking_plot(fl_category_utility, fl_category_entailment, fl_category_exposure, "fl_box_plot")

In [ ]:
distribution_across_ranking_plot(y_category_utility, y_category_entailment, y_category_exposure, 'y_box_plot')

In [ ]:
distribution_across_ranking_plot(cd_category_utility, cd_category_entailment, cd_category_exposure, 'cd_box_plot')

In [ ]:
distribution_across_ranking_plot(rp_category_utility, rp_category_entailment, rp_category_exposure, "rp_box_plot")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Melt helper
def melt_df(df, value_name):
    return df.reset_index().melt(id_vars='index', var_name='Group', value_name=value_name)

# Main plot function
def relationship_between_metrics_per_group__plot(category_utility, category_entailment, category_exposure, output_file_name):
    # Prepare long data
    utility_long = melt_df(category_utility, 'Utility')
    consumption_long = melt_df(category_entailment, 'Consumption')
    exposure_long = melt_df(category_exposure, 'Exposure')

    # Merge all
    merged = utility_long.merge(consumption_long, on=['index', 'Group'])
    merged = merged.merge(exposure_long, on=['index', 'Group'])
    merged.rename(columns={'index': 'Ranking'}, inplace=True)

    # Pairs to plot
    pairs = [
        ('Exposure', 'Consumption', 'Exposure vs Consumption'),
        ('Consumption', 'Utility', 'Consumption vs Utility'),
    ]

    # Markers & colors
    group_markers = ['o', 's', 'D', '^']
    palette = sns.color_palette("colorblind")
    groups = merged['Group'].unique()

    # Iterate over each pair
    for x_var, y_var, title in pairs:
        fig, axes = plt.subplots(1, len(groups), figsize=(20, 5), sharex=False, sharey=False)

        for i, group in enumerate(groups):
            data = merged[merged['Group'] == group]
            ax = axes[i]

            sns.scatterplot(
                data=data,
                x=x_var,
                y=y_var,
                ax=ax,
                color=palette[i],
                marker=group_markers[i],
                s=80,
                edgecolor='k',
                alpha=0.8
            )

            ax.set_title(f'{title}\nGroup: {group}')
            ax.set_xlabel(x_var)
            ax.set_ylabel(y_var)

        plt.tight_layout()
        output_path = f'{output_file_name}_scatter_{x_var}_{y_var}.png'
        plt.savefig(output_path, dpi=300)  # High-resolution PNG
        plt.close()  # Close to avoid displaying inline if not needed


In [ ]:
relationship_between_metrics_per_group__plot(fl_category_utility, fl_category_entailment, fl_category_exposure, "fl")

In [ ]:
relationship_between_metrics_per_group__plot(y_category_utility, y_category_entailment, y_category_exposure, "y")

In [ ]:
relationship_between_metrics_per_group__plot(cd_category_utility, cd_category_entailment, cd_category_exposure, "cd")

In [ ]:
relationship_between_metrics_per_group__plot(rp_category_utility, rp_category_entailment, rp_category_exposure, "rp")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def ration_of_consumption_to_utility_plot(category_utility, category_entailment, output_file_name):
    # Step 1: Compute ratio DataFrame
    ratio_df = category_entailment / category_utility

    # Step 2: Convert to long format for plotting
    ratio_long = ratio_df.reset_index().melt(id_vars='index', var_name='Group', value_name='Consumption_to_Utility_Ratio')
    ratio_long.rename(columns={'index': 'Ranking'}, inplace=True)

    # Color-blind palette
    palette = sns.color_palette("colorblind")

    # Step 3: Boxplot — distribution per group
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=ratio_long, x='Group', y='Consumption_to_Utility_Ratio', palette=palette)
    plt.title("Distribution of Consumption-to-Utility Ratio per Group")
    plt.ylabel("Consumption / Utility")
    plt.tight_layout()
    plt.savefig(f"{output_file_name}.png", dpi=300)  # High-resolution PNG
    plt.close()  # Close to avoid displaying inline if not needed

In [ ]:
ration_of_consumption_to_utility_plot(fl_category_utility, fl_category_entailment, "fl_box_plot")

In [ ]:
ration_of_consumption_to_utility_plot(y_category_utility, y_category_entailment, "y_box_plot")

In [ ]:
ration_of_consumption_to_utility_plot(cd_category_utility, cd_category_entailment, "cd_box_plot")

In [ ]:
ration_of_consumption_to_utility_plot(rp_category_utility, rp_category_entailment, "rp_box_plot")

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt

# # Optional: shorten ranking axis if too long
# def plot_stacked_bar(df, title, ylabel):
#     df = df.copy()
#     df.index.name = 'Ranking'
#     df = df.reset_index()

#     # Set 'Ranking' as string for better x-axis labeling
#     df['Ranking'] = df['Ranking'].astype(str)
#     df.set_index('Ranking', inplace=True)

#     # Plot
#     ax = df.plot(
#         kind='bar',
#         stacked=True,
#         figsize=(12, 6),
#         colormap='tab20'  # colorblind-friendly palette
#     )
#     ax.set_title(title)
#     ax.set_ylabel(ylabel)
#     ax.set_xlabel("Ranking")
#     ax.legend(title="Group", bbox_to_anchor=(1.05, 1), loc='upper left')
#     plt.tight_layout()
#     plt.show()

In [ ]:
# # Call the function for each metric
# plot_stacked_bar(fl_category_utility, "Group Contribution to Total Utility per Ranking", "Utility")
# plot_stacked_bar(fl_category_entailment, "Group Contribution to Total Consumption per Ranking", "Consumption")
# plot_stacked_bar(fl_category_exposure, "Group Contribution to Total Exposure per Ranking", "Exposure")

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# def plot_heatmap(df, title):
#     plt.figure(figsize=(12, 8))
#     sns.heatmap(
#         df,
#         cmap='viridis',   # good perceptual colormap; alternative: 'magma', 'plasma'
#         cbar_kws={'label': title},
#         linewidths=0.5,
#         linecolor='gray'
#     )
#     plt.title(title)
#     plt.xlabel('Group')
#     plt.ylabel('Ranking')
#     plt.tight_layout()
#     plt.show()

In [ ]:
# # Plot heatmaps for each metric
# plot_heatmap(fl_category_utility, "Utility Heatmap")
# plot_heatmap(fl_category_entailment, "Consumption Heatmap")
# plot_heatmap(fl_category_exposure, "Exposure Heatmap")

# test codes

In [ ]:
premise = "The quick brown fox jumps over the lazy dog."
hypotheses = ["The fox is almost quick."]
result = classifier(premise, hypotheses, multi_label=False)
print(result)

{'sequence': 'The quick brown fox jumps over the lazy dog.', 'labels': ['The fox is almost quick.'], 'scores': [0.9858929514884949]}


In [ ]:
!du -sh Diaz-RAG/

8.9G	Diaz-RAG/


In [ ]:
%cd Diaz-RAG/

/content/drive/MyDrive/Colab Notebooks/Diaz-RAG


In [ ]:
import json

# Open and read the JSON file
with open('test_questions.json', 'r') as f:
    data = json.load(f)

In [ ]:
for i, item in enumerate(data):
    if i == 3:
        break
    print(f"{item}")

In [ ]:
with open('test_questions.json', 'r') as f:
    first_line = f.readline()
    print(first_line)

Buffered data was truncated after reaching the output size limit.